# BiLSTM + GAT — POA v2

Notebook allineato a `main.py` e a `docs/BILSTM_GAT_STATE.md`. La modalità POA-on/off viene importata dalle costanti del branch corrente.

In [ ]:
import json
import os
import sys
from pathlib import Path

import torch
import wandb

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from main import (
    CHECKPOINT_DIR_BASE,
    FEATURE_SET,
    INCLUDE_POA_INPUTS,
    SEQ_LEN_MODEL,
    _normalize_dataset,
    _resolve_data_paths,
)
from physiq_pv.data.dataset import N_FEATURES
from physiq_pv.data.sentinel_hourly_loader import (
    load_sentinel_hourly,
    merge_with_weather,
)
from train import train

In [ ]:
DATA_PATHS = _resolve_data_paths(ROOT)

CONFIG = {
    "year": 2019,
    **DATA_PATHS,
    "source_timezone": "Europe/Rome",
    "seeds": [42, 123, 2024],
    "n_epochs": 15,
    "batch_size": 8,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "num_workers": 4,
    "early_stopping_patience": 5,
    "early_stopping_min_delta": 1e-4,
    "lam": 0.1,
    "peak_alpha": 2.5,
    "peak_gamma": 2.0,
    "peak_loss_weight": 0.25,
    "under_penalty": 3.0,
    "pr_max": 1.5,
    "night_loss_weight": 0.2,
    "validation_fraction": 0.2,
    "split_strategy": "monthly_80_20",
    "forecast_horizon": 1,
    "selection_metric": "rmse_pv_day",
    "poa_kt_max": 1.6,
    "graph_max_dist_km": 20.0,
    "graph_distance_scale_km": None,
    "edge_prior_strength": 1.0,
    "d_model": 128,
    "gat_dim": 96,
    "gat_heads": 4,
    "gat_layers": 1,
    "dropout": 0.2,
    "bilstm_pooling": "attn",
    "use_wandb": True,
    "wandb_entity": "albertopedalino-politecnico-di-torino",
    "wandb_project": "physiq_pv",
}
CONFIG

In [ ]:
ds = load_sentinel_hourly(
    sentinel_dir=CONFIG["sentinel_dir"],
    year=CONFIG["year"],
    plant_mapping_path=(
        None
        if CONFIG["plant_mapping_path"] is None
        else str(CONFIG["plant_mapping_path"])
    ),
    energy_coords_path=(
        None
        if CONFIG["energy_coords_path"] is None
        else str(CONFIG["energy_coords_path"])
    ),
    source_timezone=CONFIG["source_timezone"],
)
ds = merge_with_weather(ds, pvgis_path=str(CONFIG["pvgis_path"]))
ds = _normalize_dataset(ds)
print(ds.sizes, list(ds.data_vars))

In [ ]:
mode_tags = (
    ["poa-clear-sky"]
    if INCLUDE_POA_INPUTS
    else ["poa-input-ablation", "poa-clear-sky-target"]
)
SWEEP_CONFIG = {
    "name": (
        f"{FEATURE_SET}_{CONFIG['split_strategy']}"
        f"_h{CONFIG['forecast_horizon']}_seed_grid"
    ),
    "method": "grid",
    "metric": {
        "name": CONFIG["selection_metric"],
        "goal": "minimize",
    },
    "parameters": {
        "seed": {"values": CONFIG["seeds"]},
    },
}
seed_summary = []

def run_sweep_member():
    with wandb.init() as run:
        seed = int(run.config["seed"])
        protocol = (
            f"{CONFIG['split_strategy']}_h{CONFIG['forecast_horizon']}"
        )
        checkpoint_dir = Path(
            f"{ROOT / CHECKPOINT_DIR_BASE}_{protocol}"
            f"_pool{CONFIG['bilstm_pooling']}_seed{seed}"
        )
        run_name = (
            f"{FEATURE_SET}_f{N_FEATURES}_seq{SEQ_LEN_MODEL}"
            f"_{protocol}"
            f"_pool{CONFIG['bilstm_pooling']}_seed{seed}"
        )
        tags = [
            "bilstm-gat",
            "train-only-preprocessing",
            FEATURE_SET,
            *mode_tags,
            f"seq_len_{SEQ_LEN_MODEL}",
            f"seed_{seed}",
            "seed_grid_sweep",
            CONFIG["split_strategy"],
            f"horizon_{CONFIG['forecast_horizon']}h",
        ]
        run.name = run_name
        run.tags = tuple(tags)

        model, train_history, val_history, edge_index, edge_weight, best_epoch = train(
            ds=ds,
            n_epochs=CONFIG["n_epochs"],
            lam=CONFIG["lam"],
            early_stopping_patience=CONFIG["early_stopping_patience"],
            early_stopping_min_delta=CONFIG["early_stopping_min_delta"],
            peak_alpha=CONFIG["peak_alpha"],
            peak_gamma=CONFIG["peak_gamma"],
            peak_loss_weight=CONFIG["peak_loss_weight"],
            under_penalty=CONFIG["under_penalty"],
            pr_max=CONFIG["pr_max"],
            night_loss_weight=CONFIG["night_loss_weight"],
            validation_fraction=CONFIG["validation_fraction"],
            split_strategy=CONFIG["split_strategy"],
            forecast_horizon=CONFIG["forecast_horizon"],
            selection_metric=CONFIG["selection_metric"],
            include_poa_inputs=INCLUDE_POA_INPUTS,
            poa_kt_max=CONFIG["poa_kt_max"],
            batch_size=CONFIG["batch_size"],
            lr=CONFIG["lr"],
            weight_decay=CONFIG["weight_decay"],
            num_workers=CONFIG["num_workers"],
            graph_max_dist_km=CONFIG["graph_max_dist_km"],
            graph_distance_scale_km=CONFIG["graph_distance_scale_km"],
            edge_prior_strength=CONFIG["edge_prior_strength"],
            d_model=CONFIG["d_model"],
            gat_dim=CONFIG["gat_dim"],
            gat_heads=CONFIG["gat_heads"],
            gat_layers=CONFIG["gat_layers"],
            dropout=CONFIG["dropout"],
            use_wandb=CONFIG["use_wandb"],
            wandb_project=CONFIG["wandb_project"],
            wandb_entity=CONFIG["wandb_entity"],
            wandb_run_name=run_name,
            wandb_tags=tags,
            seq_len=SEQ_LEN_MODEL,
            checkpoint_dir=str(checkpoint_dir),
            bilstm_pooling=CONFIG["bilstm_pooling"],
            seed=seed,
        )

        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), checkpoint_dir / "model.pt")
        summary = model.training_summary
        with open(checkpoint_dir / "loss_history.json", "w", encoding="utf-8") as file:
            json.dump({"train": train_history, "val": val_history, "best_epoch": best_epoch}, file, indent=2)
        with open(checkpoint_dir / "model_config.json", "w", encoding="utf-8") as file:
            json.dump({
                "n_nodes": ds.sizes["plant"],
                "n_features": N_FEATURES,
                "seq_len": SEQ_LEN_MODEL,
                "d_model": CONFIG["d_model"],
                "gat_dim": CONFIG["gat_dim"],
                "gat_heads": CONFIG["gat_heads"],
                "gat_layers": CONFIG["gat_layers"],
                "dropout": CONFIG["dropout"],
                "use_bilstm": True,
                "use_gat": True,
                "bilstm_pooling": CONFIG["bilstm_pooling"],
                "poa_kt_max": CONFIG["poa_kt_max"],
                "edge_prior_strength": CONFIG["edge_prior_strength"],
                "include_poa_inputs": INCLUDE_POA_INPUTS,
                "seed": seed,
            }, file, indent=2)
        resolved_config = {
            key: str(value) if isinstance(value, Path) else value
            for key, value in CONFIG.items()
        }
        with open(checkpoint_dir / "training_config.json", "w", encoding="utf-8") as file:
            json.dump({
                **resolved_config,
                "feature_set": FEATURE_SET,
                "seq_len": SEQ_LEN_MODEL,
                "include_poa_inputs": INCLUDE_POA_INPUTS,
                "selection_metric": summary["selection_metric"],
                "best_selection_score": summary["best_selection_score"],
                "best_epoch": summary["best_val_epoch"],
                "checkpoint_dir": str(checkpoint_dir),
                "wandb_sweep_id": run.sweep_id,
                "wandb_run_id": run.id,
            }, file, indent=2)
        result = {
            "seed": seed,
            "best_epoch": summary["best_val_epoch"],
            "best_rmse_pv_day": summary["best_selection_score"],
            "checkpoint_dir": str(checkpoint_dir),
            "wandb_run_id": run.id,
        }
        seed_summary.append(result)
        run.summary.update(result)

sweep_id = wandb.sweep(
    sweep=SWEEP_CONFIG,
    project=CONFIG["wandb_project"],
    entity=CONFIG["wandb_entity"],
)
wandb.agent(
    sweep_id,
    function=run_sweep_member,
    count=len(CONFIG["seeds"]),
    project=CONFIG["wandb_project"],
    entity=CONFIG["wandb_entity"],
)

summary_path = Path(
    f"{ROOT / CHECKPOINT_DIR_BASE}_{CONFIG['split_strategy']}"
    f"_h{CONFIG['forecast_horizon']}_multi_seed_summary.json"
)
with open(summary_path, "w", encoding="utf-8") as file:
    json.dump({"sweep_id": sweep_id, "runs": seed_summary}, file, indent=2)

{"sweep_id": sweep_id, "runs": seed_summary}